# 37. XML Formatting: Using XML Structure in Prompts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/05-output-control/37_xml_formatting.ipynb)

**Category:** Output Control & Formatting  **Technique #:** 37  **Difficulty:** Intermediate

## 📋 Description

XML Formatting leverages the hierarchical, self-describing nature of XML to create structured prompts and responses. This technique is particularly effective with Claude, which has been trained extensively on XML-formatted data.

**When to use:**
- Complex nested data structures
- Documents with hierarchical relationships
- When working with Claude (optimized for XML)
- Integration with XML-based systems
- Multi-part prompts requiring clear sectioning

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│  XML-Structured Prompt                                      │
│  <input>                                                    │
│    <context>...</context>                                   │
│    <instructions>...</instructions>                         │
│    <examples>...</examples>                                 │
│  </input>                                                   │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  LLM Processes XML Structure                                │
│  - Understands hierarchy                                    │
│  - Preserves relationships                                  │
│  - Follows XML output format                                │
└─────────────────────────┬───────────────────────────────────┘
                          ▼
┌─────────────────────────────────────────────────────────────┐
│  XML-Structured Response                                    │
│  <output>                                                   │
│    <result>...</result>                                     │
│    <metadata>...</metadata>                                 │
│  </output>                                                  │
└─────────────────────────────────────────────────────────────┘
```

**Key Benefits:**
1. **Clear hierarchy** - Parent-child relationships are explicit
2. **Self-describing** - Tags document their own content
3. **Parseable** - Easy to extract with XML parsers
4. **Extensible** - Easy to add new elements

## ⚙️ Setup

In [ ]:
# Install required packages
!pip install openai xml.etree.ElementTree -q

import os
from getpass import getpass
from openai import OpenAI
import xml.etree.ElementTree as ET
import re

# Set up API key securely
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

def get_xml_response(prompt, model="gpt-4o-mini"):
    """Get response and extract XML content."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    return response.choices[0].message.content

def extract_xml_content(text, tag):
    """Extract content between XML tags."""
    pattern = f'<{tag}>(.*?)</{tag}>'
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).strip() if match else None

## 💡 Basic Example

In [ ]:
# Basic XML formatting example
basic_prompt = '''
Extract information from this text and format as XML:

<input>
Sarah Johnson is a Senior Product Manager at Stripe in San Francisco.
She has 8 years of experience in fintech and previously worked at PayPal.
She holds an MBA from Stanford and a BS in Computer Science from UC Berkeley.
</input>

Return the extracted information in this exact XML format:

<person>
  <name></name>
  <current_role>
    <title></title>
    <company></company>
    <location></location>
  </current_role>
  <experience>
    <years_total></years_total>
    <industry></industry>
    <previous_company></previous_company>
  </experience>
  <education>
    <degree>
      <type></type>
      <institution></institution>
    </degree>
    <degree>
      <type></type>
      <institution></institution>
    </degree>
  </education>
</person>
'''

result = get_xml_response(basic_prompt)
print("XML Output:")
print("=" * 50)
print(result)

# Extract specific fields
print("\n" + "=" * 50)
print("Extracted Fields:")
print(f"  Name: {extract_xml_content(result, 'name')}")
print(f"  Title: {extract_xml_content(result, 'title')}")
print(f"  Company: {extract_xml_content(result, 'company')}")

## 🌍 Real-World Example: Document Analysis with XML

In [ ]:
# Real-world: Legal document analysis
legal_prompt = '''
Analyze the following contract clause and extract structured information.

<contract_clause>
Section 4.2 - Confidentiality Obligations

The Recipient agrees to:
(a) maintain all Confidential Information in strict confidence;
(b) not disclose Confidential Information to any third parties except
    to those employees, agents, or consultants who have a need to know
    and are bound by confidentiality obligations no less restrictive
    than those contained herein;
(c) use Confidential Information solely for the purpose of evaluating
    the potential business relationship between the parties;
(d) notify the Disclosing Party immediately upon discovery of any
    unauthorized use or disclosure of Confidential Information.

This obligation shall survive termination of this Agreement for a
period of five (5) years.
</contract_clause>

Return the analysis in this XML format:

<legal_analysis>
  <clause_info>
    <section_number></section_number>
    <clause_title></clause_title>
    <clause_type></clause_type>
  </clause_info>
  <obligations>
    <obligation>
      <subsection></subsection>
      <description></description>
      <party_responsible></party_responsible>
    </obligation>
    <!-- Repeat for each obligation -->
  </obligations>
  <key_terms>
    <survival_period></survival_period>
    <permitted_disclosures></permitted_disclosures>
    <permitted_use></permitted_use>
  </key_terms>
  <risk_assessment>
    <risk_level></risk_level>
    <concerns></concerns>
    <recommendations></recommendations>
  </risk_assessment>
</legal_analysis>
'''

legal_result = get_xml_response(legal_prompt)
print("Legal Document Analysis (XML):")
print("=" * 50)
print(legal_result)

# Parse and display key information
print("\n" + "=" * 50)
print("Key Information:")
clause_info = extract_xml_content(legal_result, 'clause_info')
if clause_info:
    print(f"  Section: {extract_xml_content(clause_info, 'section_number')}")
    print(f"  Title: {extract_xml_content(clause_info, 'clause_title')}")

risk = extract_xml_content(legal_result, 'risk_assessment')
if risk:
    print(f"  Risk Level: {extract_xml_content(risk, 'risk_level')}")

## ❌ Failure Case: Improper XML Nesting

In [ ]:
# Failure case: Vague XML request
print("BAD EXAMPLE - Vague XML Request:")
print("=" * 50)

bad_prompt = '''
Extract information from: "John, 30, Engineer at Google"
Return as XML.
'''

bad_result = get_xml_response(bad_prompt)
print(bad_result)
print("\n❌ Problem: Inconsistent structure, may vary between runs")

print("\n" + "=" * 50)
print("GOOD EXAMPLE - Explicit XML Schema:")
print("=" * 50)

good_prompt = '''
Extract information from: "John, 30, Engineer at Google"

Return in this EXACT XML format (fill in the values):
<employee>
  <name></name>
  <age></age>
  <position></position>
  <company></company>
</employee>
'''

good_result = get_xml_response(good_prompt)
print(good_result)
print("\n✅ Success: Consistent, parseable structure")

## 📊 Benchmark: XML vs JSON vs Plain Text

In [ ]:
import time

# Benchmark comparison
test_data = """
Acme Corporation Q3 2024 Report:
Revenue: $45.2M (+12% YoY)
Net Income: $8.7M (+18% YoY)
Employees: 1,247
CEO: Jane Smith
Headquarters: Austin, Texas
"""

formats = {
    "XML": '''Extract from the text and return as XML:
<company_report>
  <company_name></company_name>
  <quarter></quarter>
  <year></year>
  <financials>
    <revenue></revenue>
    <revenue_growth></revenue_growth>
    <net_income></net_income>
    <income_growth></income_growth>
  </financials>
  <operations>
    <employees></employees>
    <ceo></ceo>
    <headquarters></headquarters>
  </operations>
</company_report>
Text: {text}''',

    "JSON": '''Extract from the text and return as JSON:
{{
  "company_name": "",
  "quarter": "",
  "year": "",
  "financials": {{
    "revenue": "",
    "revenue_growth": "",
    "net_income": "",
    "income_growth": ""
  }},
  "operations": {{
    "employees": "",
    "ceo": "",
    "headquarters": ""
  }}
}}
Text: {text}''',

    "Plain Text": '''Extract from the text and format clearly:
Text: {text}'''
}

print("BENCHMARK: XML vs JSON vs Plain Text\n")
print(f"{'Format':<15} {'Time (s)':<12} {'Parseable':<12} {'Hierarchy'}")
print("-" * 55)

for fmt, template in formats.items():
    times = []
    parseable = "No"
    hierarchy = "Flat"
    
    for _ in range(3):
        start = time.time()
        result = get_xml_response(template.format(text=test_data))
        times.append(time.time() - start)
    
    avg_time = sum(times) / len(times)
    
    if fmt == "XML":
        parseable = "Yes" if '<company_report>' in result else "No"
        hierarchy = "Nested"
    elif fmt == "JSON":
        parseable = "Yes" if result.strip().startswith('{') else "No"
        hierarchy = "Nested"
    
    print(f"{fmt:<15} {avg_time:.3f}       {parseable:<12} {hierarchy}")

print("\nKey Findings:")
print("• XML: Best for hierarchical data, verbose but clear")
print("• JSON: Compact, widely supported, good for APIs")
print("• Plain Text: Fastest but requires manual parsing")
print("• XML excels with Claude, JSON with GPT models")

## 🎮 Interactive Playground

In [ ]:
# Interactive XML template builder
def create_xml_extractor(xml_template):
    """Create a reusable XML extractor."""
    def extractor(text):
        prompt = f'''
Extract information from the following text and format it as XML.

<input>
{text}
</input>

Return XML in this exact format (fill in the values):
{xml_template}
'''
        return get_xml_response(prompt)
    return extractor

# Example: Meeting notes extractor
meeting_template = '''
<meeting>
  <title></title>
  <date></date>
  <attendees>
    <attendee>
      <name></name>
      <role></role>
    </attendee>
  </attendees>
  <agenda_items>
    <item>
      <topic></topic>
      <owner></owner>
      <status></status>
    </item>
  </agenda_items>
  <action_items>
    <action>
      <task></task>
      <assignee></assignee>
      <due_date></due_date>
    </action>
  </action_items>
</meeting>
'''

meeting_extractor = create_xml_extractor(meeting_template)

# Test with sample meeting notes
sample_notes = """
Weekly Product Review - October 15, 2024

Attendees:
- Sarah Chen (Product Manager)
- Mike Johnson (Engineering Lead)
- Lisa Wong (Design Lead)

Agenda:
1. Q4 Roadmap Review - Sarah - Completed
2. Mobile App Launch - Mike - In Progress
3. User Research Findings - Lisa - Completed

Action Items:
- Finalize API documentation (Mike, due Oct 22)
- Prepare launch marketing materials (Lisa, due Oct 20)
- Schedule user testing sessions (Sarah, due Oct 18)
"""

print("Meeting Notes Extractor - XML Output")
print("=" * 50)
meeting_xml = meeting_extractor(sample_notes)
print(meeting_xml)

# Extract specific information
print("\n" + "=" * 50)
print("Quick Summary:")
title = extract_xml_content(meeting_xml, 'title')
date = extract_xml_content(meeting_xml, 'date')
print(f"  Meeting: {title}")
print(f"  Date: {date}")

## 💡 Tips & Tricks

### Model-Specific XML Advice

**Claude (Anthropic):**
- Native XML support - use `<input>`, `<output>` tags
- Excellent at preserving XML structure
- Use `<thinking>` tags for reasoning steps

**GPT Models:**
- Good XML support but may need explicit instructions
- Use `temperature=0.1` for consistency
- Provide complete template to fill

**Gemini:**
- Decent XML handling
- May need more examples for complex structures

### Best Practices

1. **Always provide the full template** - Don't expect the model to invent structure
2. **Use meaningful tag names** - Self-documenting XML is easier to parse
3. **Close all tags properly** - Ensure your template is valid XML
4. **Handle missing data** - Specify what to do when info isn't found
5. **Use attributes sparingly** - Prefer child elements for complex data
6. **Validate output** - Use XML parsers to check well-formedness

```python
# Validation example
import xml.etree.ElementTree as ET

def validate_xml(xml_string):
    try:
        ET.fromstring(xml_string)
        return True
    except ET.ParseError:
        return False
```

## 📚 References

1. [XML Specification](https://www.w3.org/XML/)
2. [Claude XML Best Practices](https://docs.anthropic.com/)
3. [Python XML Processing](https://docs.python.org/3/library/xml.etree.elementtree.html)
4. [XPath for XML Extraction](https://www.w3schools.com/xml/xpath_intro.asp)